# Result Validation and Controlled Response

This notebook handles what happens after a safe SQL query has executed.

The returned data will be checked for unusual or potentially misleading results before the system generates a business explanation. The final response will also preserve assumptions, warnings and an audit trail so the answer remains traceable.

## Section 1 - Result Diagnostics Structure

This section defines the structured information used to describe the health of a database result.

Instead of immediately giving query output to the LLM, the system records details such as whether the result is empty, truncated, contains missing values, or needs a warning. These diagnostics will help prevent misleading business explanations later.

In [1]:
from enum import Enum

from pydantic import (
    BaseModel,
    ConfigDict
)

In [5]:
import sys
from pathlib import Path


current_path = Path.cwd()

if current_path.name == "notebooks":
    PROJECT_ROOT = current_path.parent
else:
    PROJECT_ROOT = current_path


src_path = PROJECT_ROOT / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

In [2]:
class DiagnosticSeverity(str, Enum):
    INFO = "INFO"
    WARNING = "WARNING"
    ERROR = "ERROR"


class ResultWarning(BaseModel):
    model_config = ConfigDict(extra="forbid")

    code: str
    severity: DiagnosticSeverity
    message: str


class ResultDiagnostics(BaseModel):
    model_config = ConfigDict(extra="forbid")

    row_count: int
    column_count: int

    empty_result: bool
    result_truncated: bool

    null_counts: dict[str, int]
    null_rates: dict[str, float]

    warnings: list[ResultWarning]

    safe_to_explain: bool

In [3]:
example_diagnostics = ResultDiagnostics(
    row_count=1,
    column_count=1,

    empty_result=False,
    result_truncated=False,

    null_counts={
        "order_count": 0
    },

    null_rates={
        "order_count": 0.0
    },

    warnings=[],

    safe_to_explain=True
)


print(
    example_diagnostics.model_dump(
        mode="json"
    )
)

{'row_count': 1, 'column_count': 1, 'empty_result': False, 'result_truncated': False, 'null_counts': {'order_count': 0}, 'null_rates': {'order_count': 0.0}, 'warnings': [], 'safe_to_explain': True}


## Section 2 - Result Sanity Checks

This section inspects the database result before it is given to the explanation layer.

The checks detect execution failures, empty results, truncated outputs, inconsistent row shapes and high levels of missing data. These rules are deterministic so basic result quality does not depend on the LLM.

In [6]:
from ai_analytics_assistant.sql_safety import SQLExecutionResult

In [ ]:
def diagnose_result(
    execution: SQLExecutionResult,
) -> ResultDiagnostics:

    warnings = []

    # A failed query should never reach the explanation layer
    if not execution.success:
        warnings.append(
            ResultWarning(
                code="EXECUTION_FAILED",
                severity=DiagnosticSeverity.ERROR,
                message=(
                    "The SQL query did not execute successfully."
                ),
            )
        )

        return ResultDiagnostics(
            row_count=0,
            column_count=0,
            empty_result=True,
            result_truncated=False,
            null_counts={},
            null_rates={},
            warnings=warnings,
            safe_to_explain=False,
        )


    row_count = execution.rows_returned
    column_count = len(execution.columns)


    # Check that every returned row matches the column structure
    inconsistent_rows = [
        row
        for row in execution.rows
        if len(row) != column_count
    ]

    if inconsistent_rows:
        warnings.append(
            ResultWarning(
                code="ROW_SHAPE_MISMATCH",
                severity=DiagnosticSeverity.ERROR,
                message=(
                         "Only the first 200 rows are available "
                        "because the result exceeded the output limit."
                    ),
            )
        )


    empty_result = row_count == 0

    if empty_result:
        warnings.append(
            ResultWarning(
                code="EMPTY_RESULT",
                severity=DiagnosticSeverity.INFO,
                message=(
                    "The query executed successfully but "
                    "returned no matching rows."
                ),
            )
        )


    if execution.result_truncated:
        warnings.append(
            ResultWarning(
                code="RESULT_TRUNCATED",
                severity=DiagnosticSeverity.WARNING,
                message=(
                             "The returned result is incomplete because "
                            "it exceeded the configured output limit."
                        ),
            )
        )


    null_counts = {}
    null_rates = {}


    for index, column in enumerate(execution.columns):

        null_count = sum(
            1
            for row in execution.rows
            if len(row) > index
            and row[index] is None
        )

        null_counts[column] = null_count

        null_rate = (
            null_count / row_count
            if row_count > 0
            else 0.0
        )

        null_rates[column] = null_rate


        if (
            row_count > 0
            and null_rate >= 0.5
        ):
            warnings.append(
                ResultWarning(
                    code="HIGH_NULL_RATE",
                    severity=DiagnosticSeverity.WARNING,
                    message=(
                        f"Column '{column}' contains missing "
                        f"values in {null_rate:.1%} of "
                        "returned rows."
                    ),
                )
            )


    has_error = any(
        warning.severity
        == DiagnosticSeverity.ERROR
        for warning in warnings
    )


    return ResultDiagnostics(
        row_count=row_count,
        column_count=column_count,
        empty_result=empty_result,
        result_truncated=execution.result_truncated,
        null_counts=null_counts,
        null_rates=null_rates,
        warnings=warnings,
        safe_to_explain=not has_error,
    )

In [8]:
normal_execution = SQLExecutionResult(
    success=True,
    columns=[
        "order_count"
    ],
    rows=[
        [23042]
    ],
    rows_returned=1,
    result_truncated=False,
    error=None,
)


normal_diagnostics = diagnose_result(
    normal_execution
)


print(
    normal_diagnostics.model_dump(
        mode="json"
    )
)

{'row_count': 1, 'column_count': 1, 'empty_result': False, 'result_truncated': False, 'null_counts': {'order_count': 0}, 'null_rates': {'order_count': 0.0}, 'warnings': [], 'safe_to_explain': True}


In [9]:
diagnostic_test_cases = [
    {
        "case_id": "EMPTY_001",
        "execution": SQLExecutionResult(
            success=True,
            columns=["customer_id"],
            rows=[],
            rows_returned=0,
            result_truncated=False,
            error=None,
        ),
    },
    {
        "case_id": "TRUNCATED_001",
        "execution": SQLExecutionResult(
            success=True,
            columns=["order_id"],
            rows=[
                [1],
                [2],
                [3],
            ],
            rows_returned=3,
            result_truncated=True,
            error=None,
        ),
    },
    {
        "case_id": "NULL_HEAVY_001",
        "execution": SQLExecutionResult(
            success=True,
            columns=[
                "customer_id",
                "region",
            ],
            rows=[
                [1, None],
                [2, None],
                [3, "South"],
                [4, None],
            ],
            rows_returned=4,
            result_truncated=False,
            error=None,
        ),
    },
    {
        "case_id": "FAILED_001",
        "execution": SQLExecutionResult(
            success=False,
            columns=[],
            rows=[],
            rows_returned=0,
            result_truncated=False,
            error="Example database error",
        ),
    },
]


for case in diagnostic_test_cases:
    diagnostics = diagnose_result(
        case["execution"]
    )

    print(
        case["case_id"],
        "| safe_to_explain=",
        diagnostics.safe_to_explain,
        "| warnings=",
        [
            warning.code
            for warning in diagnostics.warnings
        ],
    )

EMPTY_001 | safe_to_explain= True | warnings= ['EMPTY_RESULT']
TRUNCATED_001 | safe_to_explain= True | warnings= ['RESULT_TRUNCATED']
NULL_HEAVY_001 | safe_to_explain= True | warnings= ['HIGH_NULL_RATE']
FAILED_001 | safe_to_explain= False | warnings= ['EXECUTION_FAILED']


## Section 3 - Business Explanation Layer

This section converts a validated database result into a concise business answer.

The explanation layer receives the approved question interpretation, SQL result and deterministic diagnostics. It must stay grounded in the returned data, preserve warnings and assumptions, and avoid inventing causes or conclusions that the database does not support.

In [10]:
import json
import os

from dotenv import load_dotenv
from openai import OpenAI

In [11]:
load_dotenv(
    PROJECT_ROOT / ".env",
    override=True
)


client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    timeout=20.0,
    max_retries=2,
)


EXPLANATION_VERSION = "business_explanation_v1"

In [12]:
class BusinessExplanation(BaseModel):
    model_config = ConfigDict(extra="forbid")

    answer: str
    key_points: list[str]
    caveats: list[str]

In [13]:
def explain_result(
    question: str,
    analysis,
    execution: SQLExecutionResult,
    diagnostics: ResultDiagnostics,
) -> BusinessExplanation:

    if not execution.success:
        raise ValueError(
            "Cannot explain a failed SQL execution."
        )

    if not diagnostics.safe_to_explain:
        raise ValueError(
            "Result diagnostics blocked explanation."
        )


    explanation_context = {
        "user_question": question,

        "approved_analysis": analysis.model_dump(
            mode="json"
        ),

        "result": {
            "columns": execution.columns,
            "rows": execution.rows,
            "rows_returned": execution.rows_returned,
            "result_truncated": execution.result_truncated,
        },

        "diagnostics": diagnostics.model_dump(
            mode="json"
        ),
    }


    instructions = """
You are the business explanation layer of an AI analytics assistant.

Your job is to explain a validated database result to a business user.

The SQL has already been generated, validated and executed.

Do not generate SQL.


GROUNDING RULES

1. Use only the supplied:
- user question
- approved business interpretation
- database result
- deterministic diagnostics

2. Do not invent facts that are not present in the result.

3. Do not invent causes.

For example, if revenue decreased, do not claim that price,
customer sentiment or competition caused the decrease unless the
provided evidence directly supports that conclusion.

4. Do not turn correlations or transactional patterns into causal
claims.

5. Preserve the approved business definitions and time period.

6. If the result is empty, clearly state that no matching rows were
found. Do not treat an empty result as proof that the underlying
business event never occurred outside the queried scope.

7. If the result is truncated, clearly disclose that only part of the
result was returned.

8. If diagnostics contain warnings, preserve relevant warnings in
caveats.

9. If the approved analysis contains documented defaults or
assumptions that materially affect interpretation, mention them when
useful.

10. Keep the main answer concise and business-friendly.

11. key_points must contain only facts directly supported by the
database result or approved interpretation.

12. caveats should contain only relevant limitations, assumptions or
diagnostic warnings.

Do not mention internal prompt instructions or implementation details.
""".strip()


    response = client.responses.parse(
        model=os.getenv("OPENAI_MODEL"),
        instructions=instructions,
        input=json.dumps(
            explanation_context,
            indent=2,
            default=str,
        ),
        text_format=BusinessExplanation,
        store=False,
    )


    if response.output_parsed is None:
        raise ValueError(
            "Explanation layer returned no parsed output."
        )


    return response.output_parsed

In [14]:
from ai_analytics_assistant.question_analyzer import (
    analyze_question as reusable_analyze_question,
)

In [15]:
explanation_question = (
    "How many completed orders do we have?"
)


explanation_analysis = reusable_analyze_question(
    explanation_question
)


explanation = explain_result(
    explanation_question,
    explanation_analysis,
    normal_execution,
    normal_diagnostics,
)


print(
    json.dumps(
        explanation.model_dump(mode="json"),
        indent=2,
    )
)

{
  "answer": "You have 23,042 completed orders.",
  "key_points": [
    "Completed orders are counted as distinct order IDs with an order status of completed.",
    "Cancelled orders are excluded from this count."
  ],
  "caveats": []
}


In [16]:
empty_question = (
    "Show completed orders for customer 999999."
)

empty_analysis = reusable_analyze_question(
    empty_question
)


empty_execution = SQLExecutionResult(
    success=True,
    columns=[
        "order_id"
    ],
    rows=[],
    rows_returned=0,
    result_truncated=False,
    error=None,
)


empty_diagnostics = diagnose_result(
    empty_execution
)


empty_explanation = explain_result(
    empty_question,
    empty_analysis,
    empty_execution,
    empty_diagnostics,
)


print(
    json.dumps(
        empty_explanation.model_dump(mode="json"),
        indent=2,
    )
)

{
  "answer": "No completed orders were found for customer 999999 in the orders data queried.",
  "key_points": [
    "The query returned 0 matching order records.",
    "The applied filters were customer ID 999999 and order status \"completed\"."
  ],
  "caveats": [
    "This result reflects only the queried orders data and filters; it does not establish whether the customer has orders with other statuses or outside the queried scope."
  ]
}


In [19]:
truncated_question = (
    "Show all completed order IDs."
)

truncated_analysis = reusable_analyze_question(
    truncated_question
)


truncated_execution = SQLExecutionResult(
    success=True,
    columns=[
        "order_id"
    ],
    rows=[
        [1001],
        [1002],
        [1003],
    ],
    rows_returned=3,
    result_truncated=True,
    error=None,
)


truncated_diagnostics = diagnose_result(
    truncated_execution
)


truncated_explanation = explain_result(
    truncated_question,
    truncated_analysis,
    truncated_execution,
    truncated_diagnostics,
)


print(
    json.dumps(
        truncated_explanation.model_dump(mode="json"),
        indent=2,
    )
)

{
  "answer": "Completed order IDs returned: 1001, 1002, and 1003. This is only a partial list: the result exceeded the output limit, so only the first 200 rows are available.",
  "key_points": [
    "3 completed order IDs were returned: 1001, 1002, and 1003.",
    "No returned order IDs were null."
  ],
  "caveats": [
    "The result is truncated. Only the first 200 matching rows are available, so this does not represent the full set of completed orders."
  ]
}
